In [ ]:
!pip install transformers torch biopython pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.1 MB/s eta 0:00:00


In [ ]:
import torch
from transformers import AutoTokenizer, EsmForMaskedLM
from Bio import pairwise2
from Bio.Align import substitution_matrices
import pandas as pd

# --- SCIENTIFIC SEQUENCES ---
# DtpA (E. coli) - UniProt: P75743
dtpa_seq = "MSTANQKPTESVSLNAFKQPKAFYLIFSIELWERFGYYGLQGIMAVYLVKQLGMSEADSITLFSSFSALVYGLVAIGGWLGDKVLGTKRVIMLGAIVLAIGYTLVAWSGHDAGIVYMGMAAIAVGNGLFKANPSSLLSTCYEKNDPRLDGAFTMYYMSVNIGSFFSMIATPWLAAKYGWSVAFALSVVGLLITIVNFAFCQRWVKQYGSKPDFEPINYRNLLLTIIGVVALIAIATWLLHNQEVARMALGVVAFGIVVIFGKEAFAMKGAARRKMIVAFILMLEAIIFFVLYSQMPTSLNFFAIRNVEHSILGLAVEPEQYQALNPFWIIIGSPILAAIYNKMGDTLPMPTKFAIGMVMCSGAFLILPLGAKFASDAGIVSVSWLVASYGLQSIGELMISGLGLAMVAQLVPQRLMGFIMGSWFLTTAGANLIGGYVAGMMAVPDNVTDPLMSLEVYGRVFLQIGVATAVIAVLMLLTAPKLHRMTQDDAADKAAKAAVA" # Use full UniProt: Q5KWA5
# Human PepT1 (Target - SLC15A1)
 # Replace with full P75743 FASTA
# Human PepT1 - UniProt: P46059
hpept1_seq = "MGMSKSHSFFGYPLSIFFIVVNEFCERFSYYGMRAILILYFTNFISWDDNLSTAIYHTFVALCYLTPILGALIADSWLGKFKTIVSLSIVYTIGQAVTSVSSINDLTDHNHDGTPDSLPVHVVLSLIGLALIALGTGGIKPCVSAFGGDQFEEGQEKQRNRFFSIFYLAINAGSLLSTIITPMLRVQQCGIHSKQACYPLAFGVPAALMAVALIVFVLGSGMYKKFKPQGNIMGKVAKCIGFAIKNRFRHRSKAFPKREHWLDWAKEKYDERLISQIKMVTRVMFLYIPLPMFWALFDQQGSRWTLQATTMSGKIGALEIQPDQMQTVNAILIVIMVPIFDAVLYPLIAKCGFNFTSLKKMAVGMVLASMAFVVAAIVQVEIDKTLPVFPKGNEVQIKVLNIGNNTMNISLPGEMVTLGPMSQTNAFMTFDVNKLTRINISSPGSPVTAVTDDFKQGQRHTLLVWAPNHYQVVKDGLNQKPEKGENGIRFVNTFNELITITMSGKVYANISSYNASTYQFFPSGIKGFTISSTEIPPQCQPNFNTFYLEFGSAYTYIVQRKNDSCPEVKVFEDISANTVNMALQIPQYFLLTCGEVVFSVTGLEFSYSQAPSNMKSVLQAGWLLTVAVGNIIVLIVAGAGQFSKQWAEYILFAALLLVVCVVFAIMARFYTYINPAEIEAQFDEDEKKNRLEKSNPYFMSGANSQKQM" # Replace with full P46059 FASTA

# --- 1. SEQUENCE ALIGNMENT ---
# Using BLOSUM62 to find evolutionary equivalents
matrix = substitution_matrices.load("BLOSUM62")
alignments = pairwise2.align.globalds(dtpa_seq, hpept1_seq, matrix, -10, -0.5)
aln_dtpa, aln_hu, score, _, _ = alignments[0]

# --- 2. BINDING POCKET MAPPING ---
# Key residues in Human PepT1 known for drug binding
# (e.g., Y167, W294, E595)
human_pocket = [24, 26, 27, 31, 34, 56, 57, 60, 63, 64, 80, 75, 91, 140, 144, 162, 166, 167, 171, 282, 294, 297, 298, 328, 341, 594]
targets = []
hu_idx = 0
for i, (r_dtpa, r_hu) in enumerate(zip(aln_dtpa, aln_hu)):
    if r_hu != '-': hu_idx += 1
    if hu_idx in human_pocket and r_dtpa != '-':
        dtpa_pos = i - aln_dtpa[:i].count('-')
        targets.append({'pos': dtpa_pos, 'hu_res': r_hu, 'dtpa_res': r_dtpa})

# --- 3. ESM-2 STABILITY SCORING ---
model_name = "facebook/esm2_t33_650M_UR50D"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)
model.eval()

def get_fitness(seq, pos, mutant_res):
    inputs = tokenizer(seq, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=-1)
    token_id = tokenizer.encode(mutant_res, add_special_tokens=False)[0]
    return probs[0, pos + 1, token_id].item()

# --- 4. EXECUTION ---
print(f"Scientific Alignment Score: {score}")
results = []
for t in targets:
    fitness = get_fitness(dtpa_seq, t['pos'], t['hu_res'])
    results.append({
        'DtpA_Site': f"{t['dtpa_res']}{t['pos']}",
        'Human_Residue': t['hu_res'],
        'Stability_Score': fitness
    })

print(pd.DataFrame(results))

/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.61G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/571 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Scientific Alignment Score: 209.0
   DtpA_Site Human_Residue  Stability_Score
0        L30             F     1.611095e-04
1        E32             E     9.996504e-01
2        R33             R     9.998010e-01
3        Y37             Y     9.996258e-01
4        Q40             R     7.750429e-06
5        F62             Y     8.272395e-04
6        S63             H     3.100413e-06
7        S66             V     1.545571e-04
8        V69             C     1.252147e-03
9        Y70             Y     9.997942e-01
10       D81             D     9.997689e-01
11       T86             K     1.278857e-05
12       L97             Y     3.147948e-07
13      K129             K     9.998767e-01
14      S133             S     9.966123e-01
15      A150             F     7.227899e-07
16      Y154             F     2.993291e-02
17      Y155             Y     9.998152e-01
18      N159             N     9.996744e-01
19      A284             R     2.709767e-07
20      F301             W     3.922801e-0

In [ ]:
import torch
import pandas as pd
from transformers import AutoTokenizer, EsmForMaskedLM
from Bio import pairwise2
from Bio.Align import substitution_matrices
import warnings
from Bio import BiopythonDeprecationWarning

# Suppress Biopython deprecation warning for clean output
warnings.simplefilter('ignore', BiopythonDeprecationWarning)

# --- 1. CONFIGURATION & SEQUENCES ---
dtpa_seq = "MSTANQKPTESVSLNAFKQPKAFYLIFSIELWERFGYYGLQGIMAVYLVKQLGMSEADSITLFSSFSALVYGLVAIGGWLGDKVLGTKRVIMLGAIVLAIGYTLVAWSGHDAGIVYMGMAAIAVGNGLFKANPSSLLSTCYEKNDPRLDGAFTMYYMSVNIGSFFSMIATPWLAAKYGWSVAFALSVVGLLITIVNFAFCQRWVKQYGSKPDFEPINYRNLLLTIIGVVALIAIATWLLHNQEVARMALGVVAFGIVVIFGKEAFAMKGAARRKMIVAFILMLEAIIFFVLYSQMPTSLNFFAIRNVEHSILGLAVEPEQYQALNPFWIIIGSPILAAIYNKMGDTLPMPTKFAIGMVMCSGAFLILPLGAKFASDAGIVSVSWLVASYGLQSIGELMISGLGLAMVAQLVPQRLMGFIMGSWFLTTAGANLIGGYVAGMMAVPDNVTDPLMSLEVYGRVFLQIGVATAVIAVLMLLTAPKLHRMTQDDAADKAAKAAVA"
hpept1_seq = "MGMSKSHSFFGYPLSIFFIVVNEFCERFSYYGMRAILILYFTNFISWDDNLSTAIYHTFVALCYLTPILGALIADSWLGKFKTIVSLSIVYTIGQAVTSVSSINDLTDHNHDGTPDSLPVHVVLSLIGLALIALGTGGIKPCVSAFGGDQFEEGQEKQRNRFFSIFYLAINAGSLLSTIITPMLRVQQCGIHSKQACYPLAFGVPAALMAVALIVFVLGSGMYKKFKPQGNIMGKVAKCIGFAIKNRFRHRSKAFPKREHWLDWAKEKYDERLISQIKMVTRVMFLYIPLPMFWALFDQQGSRWTLQATTMSGKIGALEIQPDQMQTVNAILIVIMVPIFDAVLYPLIAKCGFNFTSLKKMAVGMVLASMAFVVAAIVQVEIDKTLPVFPKGNEVQIKVLNIGNNTMNISLPGEMVTLGPMSQTNAFMTFDVNKLTRINISSPGSPVTAVTDDFKQGQRHTLLVWAPNHYQVVKDGLNQKPEKGENGIRFVNTFNELITITMSGKVYANISSYNASTYQFFPSGIKGFTISSTEIPPQCQPNFNTFYLEFGSAYTYIVQRKNDSCPEVKVFEDISANTVNMALQIPQYFLLTCGEVVFSVTGLEFSYSQAPSNMKSVLQAGWLLTVAVGNIIVLIVAGAGQFSKQWAEYILFAALLLVVCVVFAIMARFYTYINPAEIEAQFDEDEKKNRLEKSNPYFMSGANSQKQM"

# Binding pocket mapping
human_pocket = [24, 26, 27, 31, 34, 56, 57, 60, 63, 64, 80, 75, 91, 140, 144, 162, 166, 167, 171, 282, 294, 297, 298, 328, 341, 594]

# Risk threshold (Scores below this trigger an epistatic rescue search)
RISK_THRESHOLD = 1e-3
SEARCH_RANGE = 3 # +/- residues to search for rescue

# --- 2. INITIALIZE ESM-2 MODEL ---
model_name = "facebook/esm2_t33_650M_UR50D"
print(f"Loading {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = EsmForMaskedLM.from_pretrained(model_name)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
model = model.to(device)

def get_fitness(sequence, target_pos, target_aa):
    """Calculates the PLM probability of an amino acid at a target position."""
    inputs = tokenizer(sequence, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.nn.functional.softmax(logits, dim=-1)
    token_id = tokenizer.encode(target_aa, add_special_tokens=False)[0]
    return probs[0, target_pos + 1, token_id].item()

def find_epistatic_rescue(base_seq, primary_pos, primary_mut, baseline_score, search_range=3):
    """Sweeps neighboring residues to find a secondary mutation that improves stability."""
    seq_list = list(base_seq)
    seq_list[primary_pos] = primary_mut
    single_mutant_seq = "".join(seq_list)

    amino_acids = "ACDEFGHIKLMNPQRSTVWY"
    best_rescue = None
    best_score = baseline_score

    start_search = max(0, primary_pos - search_range)
    end_search = min(len(base_seq), primary_pos + search_range + 1)

    for neighbor_pos in range(start_search, end_search):
        if neighbor_pos == primary_pos:
            continue

        original_neighbor_aa = single_mutant_seq[neighbor_pos]

        for aa in amino_acids:
            if aa == original_neighbor_aa:
                continue

            temp_list = list(single_mutant_seq)
            temp_list[neighbor_pos] = aa
            double_mutant_seq = "".join(temp_list)

            # Score the primary mutation in the context of the secondary mutation
            new_score = get_fitness(double_mutant_seq, primary_pos, primary_mut)

            if new_score > best_score:
                best_score = new_score
                best_rescue = {
                    'Secondary_Mutation': f"{original_neighbor_aa}{neighbor_pos+1}{aa}",
                    'New_Stability_Score': new_score,
                    'Improvement_Factor': new_score / baseline_score
                }

    return best_rescue

# --- 3. PHASE 1: ALIGNMENT & INITIAL SCORING ---
print("\n--- Running Alignment & Binding Pocket Mapping ---")
matrix = substitution_matrices.load("BLOSUM62")
alignments = pairwise2.align.globalds(dtpa_seq, hpept1_seq, matrix, -10, -0.5)
aln_dtpa, aln_hu, score, _, _ = alignments[0]

targets = []
hu_idx = 0
for i, (r_dtpa, r_hu) in enumerate(zip(aln_dtpa, aln_hu)):
    if r_hu != '-': hu_idx += 1
    if hu_idx in human_pocket and r_dtpa != '-':
        dtpa_pos = i - aln_dtpa[:i].count('-')
        targets.append({'pos': dtpa_pos, 'hu_res': r_hu, 'dtpa_res': r_dtpa})

print(f"Alignment complete. Score: {score}. Found {len(targets)} mutable targets.")

initial_results = []
for t in targets:
    fitness = get_fitness(dtpa_seq, t['pos'], t['hu_res'])
    initial_results.append({
        'DtpA_Site': f"{t['dtpa_res']}{t['pos']+1}",
        'Pos_Index': t['pos'],
        'Mutant_Res': t['hu_res'],
        'Stability_Score': fitness
    })

df_initial = pd.DataFrame(initial_results)
print("\n--- PHASE 1 RESULTS (All Mapped Pocket Mutations) ---")
print(df_initial[['DtpA_Site', 'Mutant_Res', 'Stability_Score']])

# --- 4. PHASE 2: DYNAMIC EPISTATIC RESCUE ---
# Filter dynamically based on the risk threshold
risky_mutations = [res for res in initial_results if res['Stability_Score'] < RISK_THRESHOLD]

print(f"\n--- Running Epistatic Rescue Sweep for {len(risky_mutations)} Risky Targets (Threshold < {RISK_THRESHOLD}) ---")
final_results = []

for target in risky_mutations:
    mut_name = f"{target['DtpA_Site']}{target['Mutant_Res']}"
    print(f"Analyzing {mut_name} (Initial Score: {target['Stability_Score']:.2e})...")

    rescue_data = find_epistatic_rescue(
        base_seq=dtpa_seq,
        primary_pos=target['Pos_Index'],
        primary_mut=target['Mutant_Res'],
        baseline_score=target['Stability_Score'],
        search_range=SEARCH_RANGE
    )

    if rescue_data:
        final_results.append({
            'Target_Site': mut_name,
            'Original_Score': f"{target['Stability_Score']:.2e}",
            'Best_Rescue_Mutation': rescue_data['Secondary_Mutation'],
            'New_Score': f"{rescue_data['New_Stability_Score']:.2e}",
            'Fold_Improvement': f"{rescue_data['Improvement_Factor']:.1f}x"
        })
    else:
        final_results.append({
            'Target_Site': mut_name,
            'Original_Score': f"{target['Stability_Score']:.2e}",
            'Best_Rescue_Mutation': "None found in range",
            'New_Score': "N/A",
            'Fold_Improvement': "N/A"
        })

df_final = pd.DataFrame(final_results)
print("\n--- EPISTATIC RESCUE RESULTS ---")
print(df_final.to_string(index=False))

Loading facebook/esm2_t33_650M_UR50D...


Loading weights:   0%|          | 0/571 [00:00<?, ?it/s]

EsmForMaskedLM LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     |  | 
----------------------------+------------+--+-
esm.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Using device: cuda

--- Running Alignment & Binding Pocket Mapping ---
Alignment complete. Score: 209.0. Found 26 mutable targets.

--- PHASE 1 RESULTS (All Mapped Pocket Mutations) ---
   DtpA_Site Mutant_Res  Stability_Score
0        L31          F     1.611091e-04
1        E33          E     9.996504e-01
2        R34          R     9.998010e-01
3        Y38          Y     9.996258e-01
4        Q41          R     7.750510e-06
5        F63          Y     8.272423e-04
6        S64          H     3.100348e-06
7        S67          V     1.545594e-04
8        V70          C     1.252141e-03
9        Y71          Y     9.997942e-01
10       D82          D     9.997689e-01
11       T87          K     1.278858e-05
12       L98          Y     3.147918e-07
13      K130          K     9.998767e-01
14      S134          S     9.966123e-01
15      A151          F     7.227944e-07
16      Y155          F     2.993348e-02
17      Y156          Y     9.998152e-01
18      N160          N     9.99674